# Binary benign vs malignant baseline
Prepared 14 October 2026. **Executed 14 October 2026; all three runs complete.**
Attach the original Kaggle skin-image dataset, ISIC 2019 Training Ground Truth CSV, and ISIC 2019 Training Metadata CSV. Enable a GPU.
Reconstructs and verifies the archived split, excludes AK, then trains three binary no-augmentation baselines. Test images are never evaluated.
Requires TensorFlow, pandas, NumPy and scikit-learn. A split mismatch stops execution; do not bypass that check. Save/download model archives before ending the Kaggle session.


In [23]:
#Import all necessary libraries
from pathlib import Path
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold
from hashlib import sha256
import json, gc, zipfile
import numpy as np
import tensorflow as tf
from sklearn.metrics import (classification_report, confusion_matrix,
    accuracy_score, f1_score, balanced_accuracy_score, roc_auc_score,
    average_precision_score)

In [24]:
#Define input and output paths
root = Path.cwd() / "input"

out = Path.cwd().parent /'working/binary_baseline'
out.mkdir(parents=True, exist_ok=True)

In [ ]:
#Load input files
# Reconstruct audited source split; abort if membership differs from archive.
gt_file = next(root.rglob("ISIC_2019_Training_GroundTruth.csv"))
meta_file = next(root.rglob("ISIC_2019_Training_Metadata.csv"))

gt = pd.read_csv(gt_file)
meta = pd.read_csv(meta_file)
classes = ["MEL", "NV", "BCC", "AK", "BKL", "DF", "VASC", "SCC"]

gt["image"] = gt["image"].str.upper()
meta["image"] = meta["image"].str.upper()
gt = gt.set_index("image")
meta = meta.drop_duplicates("image").set_index("image")

In [ ]:
#Load and reconstruct images
# Retain one copy per official image ID.
paths = {}
for p in sorted(root.rglob("*")):
    if p.suffix.lower() in {".jpg", ".jpeg", ".png"}:
        image_id = p.stem.upper()
        if image_id in gt.index and image_id not in paths:
            paths[image_id] = str(p)

rows = []
for image_id, path in sorted(paths.items()):
    positives = [c for c in classes if gt.at[image_id, c] == 1]
    if len(positives) != 1:
        continue
    lesion = meta.at[image_id, "lesion_id"] if image_id in meta.index else None
    group = (
        f"lesion:{lesion}"
        if pd.notna(lesion) and str(lesion).strip()
        else f"image:{image_id}"
    )
    rows.append((image_id, path, positives[0], lesion, group))

df = pd.DataFrame(rows, columns=["image_id", "path", "label", "lesion_id", "group"])
assert len(df) == 1761, f"Expected 1761 images, found {len(df)}"

In [ ]:
#Stratified sampling into train/test/validation
outer = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
trainval_idx, test_idx = next(outer.split(df, df["label"], df["group"]))
trainval, test = df.iloc[trainval_idx].copy(), df.iloc[test_idx].copy()

inner = StratifiedGroupKFold(n_splits=4, shuffle=True, random_state=42)
train_idx, val_idx = next(inner.split(
    trainval, trainval["label"], trainval["group"]
))
train = trainval.iloc[train_idx].copy()
val = trainval.iloc[val_idx].copy()

train["split"], val["split"], test["split"] = "train", "val", "test"
splits = pd.concat([train, val, test], ignore_index=True)

In [ ]:
#Verify and save the previous step
assert not (set(train.group) & set(val.group))
assert not (set(train.group) & set(test.group))
assert not (set(val.group) & set(test.group))

output = Path.cwd().parent / "artifacts" / "isic2019_verified_split.csv"
splits.to_csv(output, index=False)

print(pd.crosstab(splits["label"], splits["split"], margins=True))
print("Saved:", output)

assert sha256(splits[['image_id', 'label', 'group', 'split']].sort_values("image_id").to_csv(index=False).encode()).hexdigest() == 'b6dffb92c47baf0b24fbf66432895f87e16718977bcbd655047d36febbc61f23', "Split differs from archived baseline"

In [ ]:
#Define binary labels (malignant/bening) #Why exclude AK?
# Binary task: preserve membership, exclude AK, keep original diagnoses
MAPPING = {**dict.fromkeys(['NV','BKL','DF','VASC'], 'benign'),
           **dict.fromkeys(['MEL','BCC','SCC'], 'malignant')}


excluded = splits[splits.label == 'AK'].copy()
splits = splits[splits.label.isin(MAPPING)].copy()
splits['original_label'] = splits.label
splits['label'] = splits.original_label.map(MAPPING)
splits['target'] = splits.label.map({'benign': 0, 'malignant': 1})

In [ ]:
#Verify and save the previous step
fingerprint = sha256(splits[['image_id','label','group','split']]
    .sort_values('image_id').to_csv(index=False).encode()).hexdigest()
assert fingerprint == 'b02d516505aec91170de557bb25d341509b390aa7806547ff91015f7f47a0e9c'
assert len(splits) == 1632 and splits.image_id.is_unique
assert splits.groupby('group').split.nunique().max() == 1
splits.to_csv(out/'isic2019_binary_split.csv', index=False)
excluded.to_csv(out/'excluded_ak.csv', index=False)

In [ ]:
#Prepare training & validation dataframes
train = splits[splits.split == 'train'].copy()
val = splits[splits.split == 'val'].copy()

assert len(train) > 0
assert len(val) > 0
assert len(splits[splits.split == 'test']) > 0

In [ ]:
#Calculate class weights
class_weights = {int(k): len(train)/(2*int(v))
                 for k,v in train.target.value_counts().items()}
print(pd.crosstab(splits.label, splits.split, margins=True))
print('Class weights:', class_weights)

In [ ]:
#Check available hardware (GPU vs CPU)
#DISCUSS ON THURSDAY
#assert tf.config.list_physical_devices('GPU'), 'Enable GPU before training.'
gpus = tf.config.list_physical_devices('GPU')

if gpus:
    print("GPU detected:", gpus)
else:
    print("No compatible GPU detected. Running on CPU.")

print("TensorFlow devices:", tf.config.list_physical_devices())

In [ ]:
#Define image loading
def load_image(path, target):
    x = tf.io.decode_image(tf.io.read_file(path), channels=3,
                           expand_animations=False)
    x.set_shape([None, None, 3])
    return tf.cast(tf.image.resize(x, [128,128]), tf.float32)/255., target

In [ ]:
#Define Cell 6.2 – Define TensorFlow dataset function
def dataset(part, seed, shuffle):
    ds = tf.data.Dataset.from_tensor_slices((part.path.astype(str).to_numpy(),
                                            part.target.to_numpy('int32')))
    ds = ds.map(load_image, num_parallel_calls=tf.data.AUTOTUNE)
    if shuffle:
        ds = ds.shuffle(len(part), seed=seed, reshuffle_each_iteration=True)
    opts = tf.data.Options()
    opts.experimental_deterministic = True
    return ds.with_options(opts).batch(32).prefetch(tf.data.AUTOTUNE)

In [ ]:
#Thursday, Y IS THIS HERE???
#Build a CNN model 
def build_cnn():
    return tf.keras.Sequential([
        tf.keras.Input((128,128,3)),
        tf.keras.layers.Conv2D(32,3,padding='same',activation='relu'),
        tf.keras.layers.MaxPooling2D(),
        tf.keras.layers.Conv2D(64,3,padding='same',activation='relu'),
        tf.keras.layers.MaxPooling2D(),
        tf.keras.layers.Conv2D(128,3,padding='same',activation='relu'),
        tf.keras.layers.MaxPooling2D(),
        tf.keras.layers.Conv2D(256,3,padding='same',activation='relu'),
        tf.keras.layers.GlobalAveragePooling2D(),
        tf.keras.layers.Dense(128,activation='relu'),
        tf.keras.layers.Dropout(.35,seed=12345),
        tf.keras.layers.Dense(1,activation='sigmoid')])

In [21]:
#Run the model 
results = []
for seed in [42,43,44]:
    tf.keras.backend.clear_session()
    tf.keras.utils.set_random_seed(seed)
    model = build_cnn()
    init_hash = sha256(b''.join(w.tobytes() for w in model.get_weights())).hexdigest()
    # Preserve initial arrays for future paired binary augmentation comparisons.
    np.savez_compressed(out/f'initial_weights_seed{seed}.npz', *model.get_weights())
    model.compile(optimizer=tf.keras.optimizers.Adam(.001),
                  loss='binary_crossentropy', metrics=['accuracy'])
    run = f'binary_no_aug_seed{seed}'
    checkpoint = out/f'{run}.keras'
    train_ds, val_ds = dataset(train, seed, True), dataset(val, seed, False)
    history = model.fit(train_ds, validation_data=val_ds, epochs=25,
        class_weight=class_weights, callbacks=[
            tf.keras.callbacks.ModelCheckpoint(str(checkpoint),
                monitor='val_loss',save_best_only=True),
            tf.keras.callbacks.EarlyStopping(monitor='val_loss',patience=5,
                                             restore_best_weights=True)], verbose=2)
    best = tf.keras.models.load_model(checkpoint)
    probability = best.predict(val_ds,verbose=0).reshape(-1)
    y = val.target.to_numpy()
    pred = (probability >= .5).astype(int)
    tn,fp,fn,tp = confusion_matrix(y,pred,labels=[0,1]).ravel()
    row = dict(seed=seed,condition='binary_no_aug',
        epochs_trained=len(history.history['loss']),
        selected_epoch=int(np.argmin(history.history['val_loss'])+1),
        validation_loss=float(min(history.history['val_loss'])),
        accuracy=float(accuracy_score(y,pred)),
        macro_f1=float(f1_score(y,pred,labels=[0,1],average='macro',zero_division=0)),
        balanced_accuracy=float(balanced_accuracy_score(y,pred)),
        malignant_sensitivity=float(tp/(tp+fn)), specificity=float(tn/(tn+fp)),
        roc_auc=float(roc_auc_score(y,probability)),
        average_precision=float(average_precision_score(y,probability)),
        threshold=.5,initial_weights_sha256=init_hash,
        split_fingerprint=fingerprint,test_evaluated=False)
    report=classification_report(y,pred,labels=[0,1],
        target_names=['benign','malignant'],output_dict=True,zero_division=0)
    (out/f'{run}_metrics.json').write_text(json.dumps(
        dict(**row,classification_report=report),indent=2))
    pd.DataFrame(history.history).assign(
        epoch=lambda d: np.arange(1,len(d)+1)).to_csv(out/f'{run}_history.csv',index=False)
    predictions=val[['image_id','original_label','label','group']].copy()
    predictions['malignant_probability']=probability
    predictions['prediction']=pred
    predictions.to_csv(out/f'{run}_validation_predictions.csv',index=False)
    pd.DataFrame(confusion_matrix(y,pred,labels=[0,1]),
        index=['benign','malignant'],columns=['benign','malignant']).to_csv(
            out/f'{run}_confusion_matrix.csv')
    results.append(row)
    pd.DataFrame(results).to_csv(out/'binary_baseline_results.csv',index=False)
    # Checkpoint each seed's complete evidence before moving to the next run.
    
    # Checkpoint each seed's complete evidence before moving to the next run.
archive_path = out / f'{run}_artifacts.zip'

with zipfile.ZipFile(archive_path, 'w',
                     zipfile.ZIP_DEFLATED) as archive:
    for p in out.glob(f'{run}*'):
        archive.write(p, p.name)

    for name in [f'initial_weights_seed{seed}.npz',
                 'isic2019_binary_split.csv']:
        archive.write(out / name, name)

print("Archive saved:", archive_path)
print('BINARY_RUN_COMPLETE',json.dumps(row),flush=True)
del model,best,train_ds,val_ds
gc.collect()

Epoch 1/25
31/31 - 19s - 597ms/step - accuracy: 0.5015 - loss: 0.6941 - val_accuracy: 0.4526 - val_loss: 0.6940
Epoch 2/25
31/31 - 19s - 606ms/step - accuracy: 0.4505 - loss: 0.6936 - val_accuracy: 0.4526 - val_loss: 0.6928
Epoch 3/25
31/31 - 17s - 536ms/step - accuracy: 0.5107 - loss: 0.6931 - val_accuracy: 0.5413 - val_loss: 0.6930
Epoch 4/25
31/31 - 16s - 516ms/step - accuracy: 0.4719 - loss: 0.6932 - val_accuracy: 0.5596 - val_loss: 0.6928
Epoch 5/25
31/31 - 15s - 484ms/step - accuracy: 0.4862 - loss: 0.6932 - val_accuracy: 0.4526 - val_loss: 0.6948
Epoch 6/25
31/31 - 15s - 499ms/step - accuracy: 0.4985 - loss: 0.6945 - val_accuracy: 0.5474 - val_loss: 0.6926
Epoch 7/25
31/31 - 16s - 530ms/step - accuracy: 0.5036 - loss: 0.6931 - val_accuracy: 0.4557 - val_loss: 0.6930
Epoch 8/25
31/31 - 15s - 492ms/step - accuracy: 0.5342 - loss: 0.6941 - val_accuracy: 0.5474 - val_loss: 0.6920
Epoch 9/25
31/31 - 15s - 489ms/step - accuracy: 0.5526 - loss: 0.6932 - val_accuracy: 0.5474 - val_loss:

KeyboardInterrupt: 

In [22]:
#Save everything
config=dict(mapping=MAPPING,class_weights=class_weights,seeds=[42,43,44],
    tensorflow_version=tf.__version__,split_fingerprint=fingerprint,
    threshold=.5,test_evaluated=False,max_epochs=25,patience=5,
    batch_size=32,steps_per_epoch=31,learning_rate=.001)
(out/'binary_baseline_config.json').write_text(json.dumps(config,indent=2))
print('ALL_BINARY_BASELINES_COMPLETE')
print(pd.DataFrame(results).to_string(index=False))
# No test image dataset is constructed or evaluated in this script.


ALL_BINARY_BASELINES_COMPLETE
 seed     condition  epochs_trained  selected_epoch  validation_loss  accuracy  macro_f1  balanced_accuracy  malignant_sensitivity  specificity  roc_auc  average_precision  threshold                                           initial_weights_sha256                                                split_fingerprint  test_evaluated
   42 binary_no_aug              13               8         0.691995  0.547401  0.353755                0.5                    0.0          1.0 0.445304           0.425427        0.5 f0b9a0ddce8f8452c7308e6e7ae6c7d37f9d3b7eb850da4831cbb7f2651b3953 b02d516505aec91170de557bb25d341509b390aa7806547ff91015f7f47a0e9c           False
